# Stage 3 CA-Only Experiment Analysis

Loads Stage 3 and Stage 2 dynamics pickles to analyze whether the frozen-SA + trainable-CA
architecture adds genuine classification signal beyond Stage 2 (SA alone).

**Null hypothesis:** CA weights stay near their initial near-zero values; adding CA does not
improve final p_correct. If refuted, CA found genuine cascade signal.

**Key plots:**
1. SA vs CA entropy trajectories (2x5 grid) — SA should be flat (frozen); CA shows learning or stays high
2. CA weight norm trajectory — does it grow from the initial ~0.01 level?
3. Stage 2 vs Stage 3 learnability scatter — above diagonal = CA helped
4. Delta p_correct bar chart — per-cytokine improvement from adding CA
5. CA entropy at convergence bar chart — diffuse = cascade; focused = direct
6. Statistical tests

In [ ]:
import pickle
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from scipy.stats import mannwhitneyu, spearmanr

from cytokine_mil.analysis.dynamics import aggregate_to_donor_level, rank_cytokines_by_learnability

STAGE3_DIR = Path("../results/stage3_ca_seed42_XXXXXXXX/")  # update with actual path
STAGE2_DIR = Path("../results/bootstrap_seed42_XXXXXXXX/")  # update with actual path

VAL_DONORS = ["Donor2", "Donor3"]

In [ ]:
with open(STAGE3_DIR / "dynamics_stage3.pkl", "rb") as f:
    dyn3 = pickle.load(f)
with open(STAGE2_DIR / "dynamics_stage2.pkl", "rb") as f:
    dyn2 = pickle.load(f)
with open(STAGE3_DIR / "cytokine_groups.json") as f:
    groups = json.load(f)

SIMPLE_CYTOKINES = groups["simple_cytokines"]
COMPLEX_CYTOKINES = groups["complex_cytokines"]
SUBSET_CYTOKINES = SIMPLE_CYTOKINES + COMPLEX_CYTOKINES

epochs3 = dyn3["logged_epochs"]
epochs2 = dyn2["logged_epochs"]

print(f"Stage 3 logged epochs: {len(epochs3)}  ({epochs3[0]}..{epochs3[-1]})")
print(f"Stage 2 logged epochs: {len(epochs2)}  ({epochs2[0]}..{epochs2[-1]})")
print(f"Simple:  {SIMPLE_CYTOKINES}")
print(f"Complex: {COMPLEX_CYTOKINES}")
print(f"Train records: {len(dyn3['records'])}  |  Val records: {len(dyn3['val_records'])}")

## 1. SA vs CA Entropy Trajectories (2x5 grid)

SA is frozen — its entropy should be flat across all epochs. CA entropy reflects whether
the trainable CA layer focused (low entropy) or remained diffuse (high entropy).

Metric: H(a) = -sum_i a_i log(a_i) for SA layer (blue, frozen) and CA layer (orange, trainable).
Mean across pseudo-tubes per cytokine.

In [ ]:
def _extract_layer_entropy(records, cytokine):
    """Return SA and CA entropy curves (one array per tube) for a cytokine."""
    sa_curves, ca_curves = [], []
    for rec in records:
        if rec["cytokine"] != cytokine:
            continue
        sa_traj = rec.get("entropy_trajectory")
        ca_traj = rec.get("entropy_trajectory_ca")
        if sa_traj is None or ca_traj is None:
            continue
        sa_curves.append(np.array(sa_traj))
        ca_curves.append(np.array(ca_traj))
    return sa_curves, ca_curves


all_cyts = SIMPLE_CYTOKINES + COMPLEX_CYTOKINES
group_map = {c: "SIMPLE" for c in SIMPLE_CYTOKINES}
group_map.update({c: "COMPLEX" for c in COMPLEX_CYTOKINES})

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharey=True)
axes = axes.flatten()

for ax, cyt in zip(axes, all_cyts):
    group = group_map.get(cyt, "")
    sa_curves, ca_curves = _extract_layer_entropy(dyn3["records"], cyt)
    if sa_curves:
        mean_sa = np.mean(sa_curves, axis=0)
        mean_ca = np.mean(ca_curves, axis=0)
        ax.plot(epochs3, mean_sa, color="steelblue",  linewidth=2, label="SA (frozen)")
        ax.plot(epochs3, mean_ca, color="darkorange", linewidth=2, linestyle="--",
                label="CA (trainable)")
    ax.set_title(f"{cyt}\n({group})", fontsize=9)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("H(a) [nats]")
    ax.legend(fontsize=7)

fig.suptitle(
    "Stage 3 CA-only — SA vs CA attention entropy\n"
    "Metric: H(a) = -sum_i a_i log(a_i). SA frozen (flat); CA changes only if it learned signal.\n"
    "Mean across pseudo-tubes per cytokine.",
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

## 2. CA Weight Norm Trajectory

If CA stays near its initial near-zero norm, the null hypothesis holds — CA added nothing.
If it grows substantially, CA learned from the classification objective.

Metric: L2 norm of [V_ca.weight, V_ca.bias, w_ca.weight, U_ca.weight] concatenated.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
epochs_all = list(range(1, len(dyn3["ca_weight_norm_trajectory"]) + 1))
ax.plot(epochs_all, dyn3["ca_weight_norm_trajectory"], color="darkviolet", lw=2)
ax.axhline(dyn3["ca_weight_norm_trajectory"][0], color="gray", ls="--", lw=0.8,
           label="Initial norm")
ax.set_xlabel("Epoch")
ax.set_ylabel("L2 norm of CA parameters")
ax.set_title(
    "CA weight norm trajectory\n"
    "Null hypothesis: norm stays near initial value (CA contributes nothing)"
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.suptitle(
    "Metric: L2 norm of [V_ca.weight, V_ca.bias, w_ca.weight, U_ca.weight] concatenated",
    fontsize=8,
)
plt.tight_layout()
plt.show()

## 3. Stage 2 vs Stage 3 Learnability Scatter

X = Stage 2 learnability AUC (SA only); Y = Stage 3 learnability AUC (SA + CA).
Points above the diagonal = CA helped classification for that cytokine.
Points on/below diagonal = no improvement (null hypothesis holds for that cytokine).

In [ ]:
donor_traj2 = aggregate_to_donor_level(dyn2["records"])
donor_traj3 = aggregate_to_donor_level(dyn3["records"])
learn2 = rank_cytokines_by_learnability(donor_traj2, exclude=["PBS"])
learn3 = rank_cytokines_by_learnability(donor_traj3, exclude=["PBS"])
auc2 = {c: a for c, a in learn2["ranking"]}
auc3 = {c: a for c, a in learn3["ranking"]}

group_color  = {"SIMPLE": "steelblue", "COMPLEX": "tomato"}
group_marker = {"SIMPLE": "o",         "COMPLEX": "^"}

fig, ax = plt.subplots(figsize=(8, 8))
all_vals = []
for cyt in SUBSET_CYTOKINES:
    if cyt not in auc2 or cyt not in auc3:
        continue
    group = "SIMPLE" if cyt in SIMPLE_CYTOKINES else "COMPLEX"
    ax.scatter(auc2[cyt], auc3[cyt],
               color=group_color[group], marker=group_marker[group], s=90, zorder=3)
    ax.annotate(cyt, (auc2[cyt], auc3[cyt]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
    all_vals.extend([auc2[cyt], auc3[cyt]])

lim = [min(all_vals) * 0.95, max(all_vals) * 1.05]
ax.plot(lim, lim, "k--", lw=0.8, label="No change diagonal")
ax.set_xlabel(
    "Stage 2 learnability AUC\nAUC(mean p_correct_trajectory), aggregated to donor level",
    fontsize=9,
)
ax.set_ylabel(
    "Stage 3 learnability AUC\nAUC(mean p_correct_trajectory) with frozen SA + CA added",
    fontsize=9,
)
ax.set_title(
    "Stage 2 vs Stage 3 learnability — does adding CA improve classification?\n"
    "Above diagonal = CA helped; below = CA hurt"
)

legend_handles = [
    plt.scatter([], [], color=group_color["SIMPLE"],  marker=group_marker["SIMPLE"],
                s=60, label="SIMPLE"),
    plt.scatter([], [], color=group_color["COMPLEX"], marker=group_marker["COMPLEX"],
                s=60, label="COMPLEX"),
]
ax.legend(handles=legend_handles + [plt.Line2D([0], [0], color="k", ls="--", lw=0.8,
                                                label="No change diagonal")],
          fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Delta p_correct Bar Chart

Final p_correct: mean over last 3 logged epochs, aggregated to donor level
(median per donor, mean across donors).

Positive delta = CA improved final classification for that cytokine.
Zero or negative = null hypothesis holds.

In [ ]:
def _final_p(records, last_n=3):
    """
    Final p_correct: mean over last_n logged epochs, aggregated to donor level
    (median per donor, mean across donors).
    """
    by_cyt_donor = defaultdict(lambda: defaultdict(list))
    for r in records:
        by_cyt_donor[r["cytokine"]][r["donor"]].append(r["p_correct_trajectory"])
    result = {}
    for cyt, by_donor in by_cyt_donor.items():
        donor_vals = [
            np.median([np.mean(t[-last_n:]) for t in trajs])
            for trajs in by_donor.values()
        ]
        result[cyt] = np.mean(donor_vals)
    return result


fp2 = _final_p(dyn2["records"])
fp3 = _final_p(dyn3["records"])
delta = {
    cyt: fp3.get(cyt, 0) - fp2.get(cyt, 0)
    for cyt in SUBSET_CYTOKINES
    if cyt in fp2 and cyt in fp3
}

fig, ax = plt.subplots(figsize=(12, 4))
cyts_sorted = sorted(delta, key=lambda c: delta[c], reverse=True)
colors = ["steelblue" if c in SIMPLE_CYTOKINES else "tomato" for c in cyts_sorted]
x = np.arange(len(cyts_sorted))
ax.bar(x, [delta[c] for c in cyts_sorted], color=colors, alpha=0.8)
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_xticks(x)
ax.set_xticklabels(cyts_sorted, rotation=35, ha="right", fontsize=9)
ax.set_ylabel(
    "Delta final p_correct (Stage 3 - Stage 2)\n"
    "aggregated to donor level (median per donor, mean across donors)"
)
ax.set_title(
    "Does adding CA improve final classification per cytokine?\n"
    "Positive = CA helped; zero/negative = null hypothesis holds"
)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 5. CA Entropy at Convergence Bar Chart

Mean CA entropy over the last 5 logged epochs per cytokine.

High entropy = CA distributes broadly (diffuse/cascade signal).
Low entropy = CA focuses on specific cells (direct signal).

Metric: H(a_CA) = -sum_i a_CA_i * log(a_CA_i), mean over last 5 logged epochs,
mean across pseudo-tubes, mean across donors.

In [ ]:
def _ca_entropy_convergence(records, last_n=5):
    """
    Mean CA entropy over last_n logged epochs per cytokine.
    Aggregated to donor level (median per donor, mean across donors).
    """
    by_cyt_donor = defaultdict(lambda: defaultdict(list))
    for r in records:
        traj = r.get("entropy_trajectory_ca")
        if traj is None:
            continue
        by_cyt_donor[r["cytokine"]][r["donor"]].append(traj)
    result = {}
    for cyt, by_donor in by_cyt_donor.items():
        donor_vals = [
            np.median([np.mean(t[-last_n:]) for t in trajs])
            for trajs in by_donor.values()
        ]
        result[cyt] = np.mean(donor_vals)
    return result


ca_ent = _ca_entropy_convergence(dyn3["records"])

fig, ax = plt.subplots(figsize=(12, 4))
cyts_sorted_ent = sorted(
    [c for c in SUBSET_CYTOKINES if c in ca_ent],
    key=lambda c: ca_ent[c], reverse=True,
)
colors_ent = ["steelblue" if c in SIMPLE_CYTOKINES else "tomato" for c in cyts_sorted_ent]
x = np.arange(len(cyts_sorted_ent))
ax.bar(x, [ca_ent[c] for c in cyts_sorted_ent], color=colors_ent, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(cyts_sorted_ent, rotation=35, ha="right", fontsize=9)
ax.set_ylabel(
    "CA attention entropy at convergence (nats)\n"
    "H(a_CA) = -sum_i a_CA_i * log(a_CA_i), mean over last 5 logged epochs"
)
ax.set_title(
    "CA entropy at convergence per cytokine\n"
    "High = CA distributes broadly (diffuse/cascade signal); "
    "Low = CA focuses on specific cells\n"
    "blue = SIMPLE, red = COMPLEX"
)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 6. Statistical Tests

- Mann-Whitney U (one-sided): SIMPLE vs COMPLEX learnability AUC from Stage 3.
- Spearman rho: Stage 2 vs Stage 3 learnability rankings (rank stability after CA is added).

In [ ]:
# Mann-Whitney U — Stage 3 AUCs
simple_aucs3  = [auc3[c] for c in SIMPLE_CYTOKINES  if c in auc3]
complex_aucs3 = [auc3[c] for c in COMPLEX_CYTOKINES if c in auc3]

print("Stage 3 CA-only — Hypothesis test (train donors)")
print("Metric: AUC(mean p_correct_trajectory), aggregated to donor level")
print()
print(f"  SIMPLE  AUCs: {[f'{x:.3f}' for x in sorted(simple_aucs3,  reverse=True)]}")
print(f"  COMPLEX AUCs: {[f'{x:.3f}' for x in sorted(complex_aucs3, reverse=True)]}")
print()

if simple_aucs3 and complex_aucs3:
    u_stat, p_one_sided = mannwhitneyu(simple_aucs3, complex_aucs3, alternative="greater")
    n1, n2 = len(simple_aucs3), len(complex_aucs3)
    r_rb = 1 - (2 * u_stat) / (n1 * n2)
    print(f"  Mann-Whitney U = {u_stat:.1f}")
    print(f"  One-sided p (simple > complex) = {p_one_sided:.4f}")
    print(f"  Rank-biserial r = {r_rb:.3f}")
    print(f"  Note: n=5 per group -> low power. Interpret effect size alongside p-value.")
    print()

# Spearman rho — Stage 2 vs Stage 3 ranking stability
ranking2 = learn2["ranking"]
ranking3 = learn3["ranking"]
order2 = [c for c, _ in ranking2 if c in auc3]
order3 = [c for c, _ in ranking3]
rank3_by_cyt = {c: i for i, c in enumerate(order3)}
ranks3_aligned = [rank3_by_cyt.get(c, len(order3)) for c in order2]

if len(order2) >= 2:
    rho, pval = spearmanr(range(len(order2)), ranks3_aligned)
    print("Stage 2 vs Stage 3 learnability ranking correlation")
    print(f"  Spearman rho = {rho:.3f}  (p = {pval:.3f})")
    print(f"  Stable (rho > 0.7): {rho > 0.7}")
    print("  High rho = adding CA did not reorder cytokine learnability")
    print("  Low rho = CA substantially changed which cytokines are 'learned' first")
    print()

# Per-cytokine summary table
print(f"{'Cytokine':<14}  {'AUC Stage2':>10}  {'AUC Stage3':>10}  {'Delta':>8}  Group")
print("-" * 56)
for cyt in SIMPLE_CYTOKINES + COMPLEX_CYTOKINES:
    a2 = auc2.get(cyt, float("nan"))
    a3 = auc3.get(cyt, float("nan"))
    d  = a3 - a2 if not (np.isnan(a2) or np.isnan(a3)) else float("nan")
    group = "SIMPLE" if cyt in SIMPLE_CYTOKINES else "COMPLEX"
    print(f"  {cyt:<14}  {a2:>10.3f}  {a3:>10.3f}  {d:>8.3f}  {group}")